# Cross-Industry Accelerator — Configuration
### Universal Setup for All Industry Implementations

This notebook defines the **industry configuration** used by all downstream notebooks.
Run this notebook first, then `%run` it from other notebooks to import the config.

**Supported Industries:** Healthcare, Finance, Insurance, Retail, CPG, Construction, Oil & Gas, Media, Law Firms, Telecom

---

### How It Works
1. Set `INDUSTRY` to your target industry key
2. The notebook auto-discovers CSV files from the data folder
3. Files are classified as `dim_*`, `fact_*`, or `stream_*` based on naming convention
4. All downstream notebooks inherit this configuration

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 0: LOAD ZERO TRUST SECURITY UTILITIES
# ════════════════════════════════════════════════════════════════════════
#%pip install nbformat


In [ ]:
%run ZT_Security_Utils

In [ ]:
%run Pipeline_Logger

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# START PIPELINE STEP TRACKING
# ════════════════════════════════════════════════════════════════════════

try:
    pipeline_step_start("00_Industry_Config", "Set industry, auto-discover tables, create Lakehouse")
except NameError:
    # Pipeline_Logger not available (manual run) - define no-op stubs
    def pipeline_step_start(*args, **kwargs): pass
    def pipeline_step_complete(*args, **kwargs): pass
    def log_pipeline_event(*args, **kwargs): pass
    def log_artifact_created(*args, **kwargs): pass

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 1: SELECT YOUR INDUSTRY (with ZT validation)
# ════════════════════════════════════════════════════════════════════════

INDUSTRY = "advertising"  # Change this to your target industry

# Supported values:
# "healthcare", "finance", "insurance", "retail", "cpg",
# "construction", "oil_and_gas", "media", "law_firms", "telecom",
# "advertising"

# ZT: Validate industry against supported list
INDUSTRY = validate_industry(INDUSTRY)
log_audit_event("INDUSTRY_SELECTED", INDUSTRY, "Validated against supported list")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 2: FABRIC ARTIFACT NAMES
# ════════════════════════════════════════════════════════════════════════
# These follow the naming convention from INDUSTRY_REPLICATION_GUIDE.md
# Override any value below if your workspace uses different names.

INDUSTRY_DISPLAY_NAMES = {
    "healthcare":   "Healthcare",
    "finance":      "Finance",
    "insurance":    "Insurance",
    "retail":       "Retail",
    "cpg":          "CPG",
    "construction": "Construction",
    "oil_and_gas":  "OilGas",
    "media":        "Media",
    "law_firms":    "LawFirm",
    "telecom":      "Telecom",
    "advertising":  "Advertising"
}

INDUSTRY_LABEL = INDUSTRY_DISPLAY_NAMES.get(INDUSTRY, INDUSTRY.title())

# ── Fabric Artifact Names ──────────────────────────────────────────────
LAKEHOUSE_NAME     = f"{INDUSTRY_LABEL}_Data_Bronze"
WAREHOUSE_NAME     = f"{INDUSTRY_LABEL}_Datawarehouse"
EVENTHOUSE_NAME    = f"{INDUSTRY.replace(' ', '_')}_rt_store"
EVENTSTREAM_NAME   = f"{INDUSTRY}_events"
ONTOLOGY_NAME      = f"{INDUSTRY_LABEL}DocBurdenOntology"
DATA_AGENT_NAME    = f"{INDUSTRY_LABEL}_QA_Agent"
OPS_AGENT_NAME     = f"{INDUSTRY_LABEL}_Ops_Agent"
SEMANTIC_MODEL_NAME = f"{INDUSTRY_LABEL}_DocBurden_Model"

print(f"Industry:       {INDUSTRY} ({INDUSTRY_LABEL})")
print(f"Lakehouse:      {LAKEHOUSE_NAME}")
print(f"Warehouse:      {WAREHOUSE_NAME}")
print(f"Eventhouse:     {EVENTHOUSE_NAME}")
print(f"Ontology:       {ONTOLOGY_NAME}")
print(f"Data Agent:     {DATA_AGENT_NAME}")
print(f"Ops Agent:      {OPS_AGENT_NAME}")
print(f"Semantic Model: {SEMANTIC_MODEL_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 2b: CREATE LAKEHOUSE IF IT DOESN'T EXIST
# ════════════════════════════════════════════════════════════════════════
# Works in both Fabric runtime (sempy) and local/VS Code (Azure CLI fallback)

import requests, json, time, subprocess

FABRIC_RUNTIME = False
try:
    import sempy.fabric as fabric
    FABRIC_RUNTIME = True
except ImportError:
    pass

if FABRIC_RUNTIME:
    # ── Running inside Fabric notebook ──
    workspace_id = fabric.get_workspace_id()
    items_df = fabric.list_items()
    lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]

    if not lh_matches.empty:
        print(f"✓ Lakehouse '{LAKEHOUSE_NAME}' already exists → {lh_matches.iloc[0].Id}")
    else:
        print(f"⚠ Lakehouse '{LAKEHOUSE_NAME}' not found. Creating...")
        access_token = notebookutils.credentials.getToken('pbi')
        headers = {
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json",
        }
        body = {
            "displayName": LAKEHOUSE_NAME,
            "type": "Lakehouse",
            "creationPayload": {
                "enableSchemas": True
            }
        }
        resp = requests.post(
            f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items",
            headers=headers,
            json=body,
        )

        if resp.status_code in (200, 201, 202):
            if resp.status_code == 202:
                operation_url = resp.headers.get("Location", "")
                print(f"  ⏳ Provisioning (202)... polling for completion.")
                for _ in range(30):
                    time.sleep(5)
                    poll = requests.get(operation_url, headers=headers)
                    if poll.status_code == 200:
                        status = poll.json().get("status", "")
                        if status == "Succeeded":
                            print(f"  ✓ Provisioning complete.")
                            break
                        elif status == "Failed":
                            raise RuntimeError(f"Lakehouse creation failed: {poll.json()}")
                else:
                    print("  ⚠ Timed out waiting for provisioning — check workspace manually.")
            else:
                result = resp.json()
                print(f"✅ Lakehouse '{LAKEHOUSE_NAME}' created → {result.get('id', 'N/A')}")

            items_df = fabric.list_items()
            lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]
            if not lh_matches.empty:
                print(f"✅ Lakehouse '{LAKEHOUSE_NAME}' confirmed → {lh_matches.iloc[0].Id}")
            else:
                print(f"⚠ Lakehouse created but not yet visible. Re-run this cell after a few seconds.")
        else:
            print(f"✗ Failed to create Lakehouse: {resp.status_code} — {resp.text}")
            raise RuntimeError(f"Lakehouse creation failed: {resp.status_code} {resp.text}")

else:
    # ── Running locally (VS Code) — use Azure CLI for Fabric REST API ──
    print("⚠ Not running in Fabric runtime (sempy unavailable). Using Azure CLI fallback.")
    try:
        token = subprocess.check_output(
            ["az", "account", "get-access-token", "--resource", "https://api.fabric.microsoft.com",
             "--query", "accessToken", "-o", "tsv"],
            text=True
        ).strip()
        headers = {"Authorization": f"Bearer {token}"}

        # List workspaces to find lakehouses
        ws_resp = requests.get("https://api.fabric.microsoft.com/v1/workspaces", headers=headers)
        ws_resp.raise_for_status()
        workspaces = ws_resp.json().get("value", [])

        found = False
        for ws in workspaces:
            ws_id = ws["id"]
            lh_resp = requests.get(
                f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/lakehouses",
                headers=headers
            )
            if lh_resp.status_code != 200:
                continue
            for lh in lh_resp.json().get("value", []):
                if lh["displayName"] == LAKEHOUSE_NAME:
                    print(f"✓ Lakehouse '{LAKEHOUSE_NAME}' found in workspace '{ws['displayName']}' → {lh['id']}")
                    found = True
                    break
            if found:
                break

        if not found:
            print(f"⚠ Lakehouse '{LAKEHOUSE_NAME}' not found in any workspace.")
            print("  Create it in the Fabric portal, or run this notebook inside Fabric to auto-create.")

    except FileNotFoundError:
        print("✗ Azure CLI ('az') not found. Install it or run this notebook in Fabric.")
    except subprocess.CalledProcessError:
        print("✗ Azure CLI auth failed. Run 'az login' first.")

log_audit_event("LAKEHOUSE_CHECK", LAKEHOUSE_NAME, "Ensured lakehouse exists")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 2c: HELPER FUNCTION FOR CREATING FABRIC ITEMS
# ════════════════════════════════════════════════════════════════════════

def _create_fabric_item(display_name, item_type):
    """Create a Fabric workspace item if it doesn't already exist.
    Returns the item ID if successful, None otherwise.
    """
    if not FABRIC_RUNTIME:
        print(f"  ⚠ Skipping {item_type} '{display_name}' creation (not in Fabric runtime)")
        return None
    
    # Check if item already exists
    items_df = fabric.list_items()
    matches = items_df[(items_df["Type"] == item_type) & (items_df["Display Name"] == display_name)]
    
    if not matches.empty:
        item_id = matches.iloc[0].Id
        print(f"  ✓ {item_type} '{display_name}' already exists → {item_id}")
        return item_id
    
    # Create new item
    print(f"  ... Creating {item_type} '{display_name}'...")
    access_token = notebookutils.credentials.getToken('pbi')
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }
    body = {
        "displayName": display_name,
        "type": item_type,
    }
    
    resp = requests.post(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items",
        headers=headers,
        json=body,
    )
    
    # Handle async provisioning (202) or immediate success (200/201)
    if resp.status_code == 202:
        operation_url = resp.headers.get("Location", "")
        print(f"      Provisioning... polling for completion")
        for _ in range(30):
            time.sleep(5)
            poll = requests.get(operation_url, headers=headers)
            if poll.status_code == 200:
                status = poll.json().get("status", "")
                if status == "Succeeded":
                    print(f"      Provisioning complete")
                    break
                elif status == "Failed":
                    print(f"  ✗ {item_type} creation failed: {poll.json()}")
                    return None
        else:
            print(f"  ⚠ Provisioning timed out — check workspace manually")
            return None
        
        # Refresh items list to get the new item ID
        items_df = fabric.list_items()
        matches = items_df[(items_df["Type"] == item_type) & (items_df["Display Name"] == display_name)]
        if not matches.empty:
            item_id = matches.iloc[0].Id
            print(f"  ✅ {item_type} '{display_name}' created → {item_id}")
            return item_id
    
    elif resp.status_code in (200, 201):
        item_id = resp.json().get("id", "")
        print(f"  ✅ {item_type} '{display_name}' created → {item_id}")
        return item_id
    
    else:
        print(f"  ✗ Failed to create {item_type}: {resp.status_code} — {resp.text}")
        return None

print("✓ Helper function loaded")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 2d: CREATE DATA FOLDER IN LAKEHOUSE
# ════════════════════════════════════════════════════════════════════════
# Uses ABFS path with explicit lakehouse ID to target the correct lakehouse,
# even when multiple lakehouses are attached to the notebook.

_HAS_NOTEBOOKUTILS = False
try:
    notebookutils  # noqa: F821 — defined in Fabric runtime
    _HAS_NOTEBOOKUTILS = True
except NameError:
    pass

folder_name = f"{INDUSTRY}_data"

if _HAS_NOTEBOOKUTILS and FABRIC_RUNTIME:
    # Build ABFS path targeting our specific lakehouse by ID
    items_df = fabric.list_items()
    lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]
    
    if not lh_matches.empty:
        lakehouse_id = lh_matches.iloc[0].Id
        # ABFS path format: abfss://<workspace_id>@onelake.dfs.fabric.microsoft.com/<lakehouse_id>/Files/<folder>
        abfs_folder_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files/{folder_name}"
        
        try:
            notebookutils.fs.ls(abfs_folder_path)
            print(f"✓ Folder '{folder_name}' already exists in '{LAKEHOUSE_NAME}'")
        except Exception:
            print(f"⚠ Folder '{folder_name}' not found in '{LAKEHOUSE_NAME}'. Creating...")
            try:
                notebookutils.fs.mkdirs(abfs_folder_path)
                print(f"✅ Folder '{folder_name}' created in '{LAKEHOUSE_NAME}'")
                log_audit_event("FOLDER_CREATED", folder_name, f"Created in lakehouse {lakehouse_id}")
            except Exception as e:
                print(f"✗ Failed to create folder: {e}")
                print(f"  → Create it manually in Fabric: Lakehouse '{LAKEHOUSE_NAME}' → Files → New folder → '{folder_name}'")
    else:
        print(f"⚠ Lakehouse '{LAKEHOUSE_NAME}' not found — cannot create folder")
else:
    print(f"⚠ Not in Fabric runtime. Folder creation skipped (use local datasets/ folder)")

print(f"\n📁 Upload your CSV files to: {LAKEHOUSE_NAME}/Files/{folder_name}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 2e: CREATE EVENTHOUSE AND EVENTSTREAM
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'─'*60}")
print("Creating Eventhouse and Eventstream...")
print(f"{'─'*60}")

# Create Eventhouse (KQL database is auto-created with same name)
eventhouse_id = _create_fabric_item(EVENTHOUSE_NAME, "Eventhouse")
if eventhouse_id:
    log_audit_event("EVENTHOUSE_CREATED", EVENTHOUSE_NAME, eventhouse_id)

# Create Eventstream (for event ingestion)
eventstream_id = _create_fabric_item(EVENTSTREAM_NAME, "Eventstream")
if eventstream_id:
    log_audit_event("EVENTSTREAM_CREATED", EVENTSTREAM_NAME, eventstream_id)

print(f"\n✅ Real-time analytics infrastructure ready")
print(f"   Eventhouse:  {EVENTHOUSE_NAME}")
print(f"   Eventstream: {EVENTSTREAM_NAME}")
print(f"\n📝 Next: Configure Eventstream routing in Fabric portal")
print(f"   1. Open Eventstream '{EVENTSTREAM_NAME}'")
print(f"   2. Add Custom Endpoint source (Event Hub compatible)")
print(f"   3. Add Eventhouse destination → select '{EVENTHOUSE_NAME}'")
print(f"   4. Configure routing rules (optional)")
print(f"{'─'*60}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 2f: AUTO-DISCOVER EVENTHOUSE CLUSTER URI
# ════════════════════════════════════════════════════════════════════════

EVENTHOUSE_CLUSTER_URI = ""  # Will be auto-populated below

if FABRIC_RUNTIME:
    print(f"\n{'─'*60}")
    print("Discovering Eventhouse cluster URI...")
    print(f"{'─'*60}")
    
    try:
        access_token = notebookutils.credentials.getToken('pbi')
        headers = {
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json",
        }
        
        # List KQL Databases in the workspace
        kql_resp = requests.get(
            f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/kqlDatabases",
            headers=headers
        )
        
        if kql_resp.status_code == 200:
            kql_databases = kql_resp.json().get("value", [])
            
            # Find the KQL Database matching our Eventhouse name
            # (Fabric auto-creates a KQL DB with the same name as the Eventhouse)
            kql_db = None
            for db in kql_databases:
                if db.get("displayName") == EVENTHOUSE_NAME:
                    kql_db = db
                    break
            
            if kql_db:
                kql_db_id = kql_db.get("id")
                print(f"  ✓ Found KQL Database '{EVENTHOUSE_NAME}' → {kql_db_id}")
                
                # Get database properties to extract queryServiceUri
                db_props_resp = requests.get(
                    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/kqlDatabases/{kql_db_id}",
                    headers=headers
                )
                
                if db_props_resp.status_code == 200:
                    props = db_props_resp.json().get("properties", {})
                    discovered_uri = props.get("queryServiceUri", "")
                    
                    if discovered_uri:
                        # Validate the URI format (ZT check)
                        from urllib.parse import urlparse
                        parsed = urlparse(discovered_uri)
                        if parsed.hostname and parsed.hostname.endswith('.kusto.fabric.microsoft.com'):
                            EVENTHOUSE_CLUSTER_URI = discovered_uri
                            print(f"  ✅ Cluster URI discovered: {EVENTHOUSE_CLUSTER_URI}")
                            log_audit_event("EVENTHOUSE_URI_DISCOVERED", EVENTHOUSE_CLUSTER_URI, "Auto-discovered from KQL Database properties")
                        else:
                            print(f"  ⚠ Invalid URI format: {discovered_uri}")
                            print(f"     Expected: https://*.kusto.fabric.microsoft.com")
                    else:
                        print(f"  ⚠ No queryServiceUri in database properties")
                        print(f"     The Eventhouse may still be provisioning. Re-run this cell in a few minutes.")
                else:
                    print(f"  ⚠ Could not get database properties: {db_props_resp.status_code}")
            else:
                print(f"  ⚠ KQL Database '{EVENTHOUSE_NAME}' not found")
                print(f"     Available databases: {[db.get('displayName') for db in kql_databases]}")
                print(f"     The database may still be provisioning. Re-run this cell in a few minutes.")
        else:
            print(f"  ⚠ Could not list KQL Databases: {kql_resp.status_code}")
    
    except Exception as e:
        print(f"  ✗ Error discovering Eventhouse URI: {e}")
    
    if not EVENTHOUSE_CLUSTER_URI:
        print(f"\n  💡 Manual fallback: Copy the cluster URI from Fabric portal:")
        print(f"     1. Open Eventhouse '{EVENTHOUSE_NAME}'")
        print(f"     2. Copy the URI from Settings → Properties → Query URI")
        print(f"     3. Update EVENTHOUSE_CLUSTER_URI in Step 3 below")
    
    print(f"{'─'*60}")
else:
    print(f"⚠ Not in Fabric runtime. Set EVENTHOUSE_CLUSTER_URI manually in Step 3.")

### ✅ Automated Setup Complete

The following artifacts are now created automatically:
1. ✓ Lakehouse folder: `{INDUSTRY}_data` (for CSV uploads)
2. ✓ Eventhouse: `{INDUSTRY}_rt_store` (for real-time analytics)
3. ✓ Eventstream: `{INDUSTRY}_events` (for event ingestion)
4. ✓ Eventhouse cluster URI (auto-discovered)

**Manual Steps Remaining:**
- Upload CSV files to the lakehouse `{INDUSTRY}_data` folder
- Configure Eventstream routing (Custom Endpoint → Eventhouse destination)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 3: DATA PATHS & BINDING CONFIGURATION
# ════════════════════════════════════════════════════════════════════════

# ── Resolve Lakehouse ID for ABFS paths ────────────────────────────────
# Uses explicit ABFS paths so it works even if another lakehouse is set as default.
if FABRIC_RUNTIME:
    _lh_items = fabric.list_items()
    _lh_match = _lh_items[(_lh_items["Type"] == "Lakehouse") & (_lh_items["Display Name"] == LAKEHOUSE_NAME)]
    LAKEHOUSE_ID = _lh_match.iloc[0].Id if not _lh_match.empty else ""
    LAKEHOUSE_ABFS_BASE = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}" if LAKEHOUSE_ID else ""
else:
    LAKEHOUSE_ID = ""
    LAKEHOUSE_ABFS_BASE = ""

# Path where CSV files are uploaded in the Lakehouse Files area
# Uses ABFS path in Fabric (works regardless of default lakehouse)
# Falls back to /lakehouse/default/ mount path for Spark reads if needed
CSV_BASE_PATH = f"/lakehouse/default/Files/{INDUSTRY}_data"  # mount path (for Spark reads when LH is default)
CSV_ABFS_PATH = f"{LAKEHOUSE_ABFS_BASE}/Files/{INDUSTRY}_data" if LAKEHOUSE_ABFS_BASE else ""  # explicit ABFS path

# Lakehouse schema (use 'dbo' for schema-enabled lakehouses)
LAKEHOUSE_SCHEMA = "dbo"

# Warehouse schema
WAREHOUSE_SCHEMA = "dbo"

# Eventhouse connection (auto-discovered in Step 2f, or set manually)
# EVENTHOUSE_CLUSTER_URI is now set automatically above
# To override, uncomment and set manually: EVENTHOUSE_CLUSTER_URI = "https://..."
# In Fabric, the auto-created KQL Database has the same name as the Eventhouse
EVENTHOUSE_DATABASE = EVENTHOUSE_NAME

# Ontology package path (if using .iq package)
ONTOLOGY_PACKAGE_PATH = f"{LAKEHOUSE_ABFS_BASE}/Files/{INDUSTRY}_ontology_package.iq" if LAKEHOUSE_ABFS_BASE else f"/lakehouse/default/Files/{INDUSTRY}_ontology_package.iq"

print(f"Lakehouse ID:  {LAKEHOUSE_ID or '(not found)'}")
print(f"CSV source:    {CSV_ABFS_PATH or CSV_BASE_PATH}")
print(f"LH schema:     {LAKEHOUSE_SCHEMA}")
print(f"WH schema:     {WAREHOUSE_SCHEMA}")
print(f"Eventhouse:    {EVENTHOUSE_CLUSTER_URI or '(not configured)'}")
print(f"Ontology pkg:  {ONTOLOGY_PACKAGE_PATH}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 4: AUTO-DISCOVER DATA SOURCES (with ZT validation)
# ════════════════════════════════════════════════════════════════════════
# Scans the CSV folder and classifies tables by naming convention:
#   dim_*    → Dimension tables  → Lakehouse + Warehouse
#   fact_*   → Fact tables       → Lakehouse + Warehouse (batch) or Eventhouse (events)
#   stream_* → Streaming tables  → Eventhouse only
#
# Works in both Fabric runtime (notebookutils) and local/VS Code (os.listdir fallback)

import os
import json

# ── Resolve CSV path: Fabric mount or local datasets/ folder ───────────
INDUSTRY_DATASET_FOLDERS = {
    "healthcare":   "healthcare_nursing_documentation",
    "finance":      "finance_banking_operations",
    "insurance":    "insurance_claims_operations",
    "retail":       "retail_store_operations",
    "cpg":          "retail_store_operations",
    "construction": "construction_site_operations",
    "oil_and_gas":  "oil_gas_field_operations",
    "media":        "media_content_operations",
    "law_firms":    "law_firm_operations",
    "telecom":      "telecom_network_operations",
    "advertising":  "advertising_campaign_operations",
}

# Detect Fabric runtime via notebookutils (more reliable than os.path.exists on FUSE)
_HAS_NOTEBOOKUTILS = False
try:
    notebookutils  # noqa: F821 — defined in Fabric runtime
    _HAS_NOTEBOOKUTILS = True
except NameError:
    pass

if _HAS_NOTEBOOKUTILS:
    # Use explicit ABFS path to target the correct lakehouse (not just default)
    if CSV_ABFS_PATH:
        _RESOLVED_CSV_PATH = CSV_ABFS_PATH
        _FS_CSV_PATH = CSV_ABFS_PATH
        print(f"✓ Fabric runtime detected — using ABFS path for '{LAKEHOUSE_NAME}'")
    else:
        _RESOLVED_CSV_PATH = CSV_BASE_PATH
        _FS_CSV_PATH = f"file:{CSV_BASE_PATH}"
        print(f"⚠ Lakehouse ID not found, falling back to default mount: {CSV_BASE_PATH}")
    
    # Diagnostic: list what's in the data folder
    try:
        _lh_files = notebookutils.fs.ls(_FS_CSV_PATH)
        _csv_names = [f.name for f in _lh_files if f.name.endswith('.csv')]
        print(f"  Found {len(_csv_names)} CSV file(s) in lakehouse path.")
        if not _csv_names:
            print(f"  ⚠ No CSV files found. Upload CSVs to {LAKEHOUSE_NAME}/Files/{INDUSTRY}_data")
    except Exception as e:
        print(f"  ⚠ Could not list data folder: {e}")
        print(f"    → Upload CSV files to lakehouse '{LAKEHOUSE_NAME}' → Files → {INDUSTRY}_data/")
else:
    # Local fallback: use datasets/<industry>/data/ relative to repo root
    _notebook_dir = os.path.dirname(os.path.abspath("__file__"))
    _repo_root = os.path.dirname(_notebook_dir) if os.path.basename(_notebook_dir) == "cross_industry_notebooks" else _notebook_dir
    _dataset_folder = INDUSTRY_DATASET_FOLDERS.get(INDUSTRY, f"{INDUSTRY}_operations")
    _RESOLVED_CSV_PATH = os.path.join(_repo_root, "datasets", _dataset_folder, "data")
    _FS_CSV_PATH = _RESOLVED_CSV_PATH
    print(f"⚠ Fabric runtime not detected. Using local fallback: {_RESOLVED_CSV_PATH}")

def _list_csv_files(base_path):
    """List CSV files — uses notebookutils in Fabric, os.listdir locally.
    For notebookutils, base_path must use the 'file:' prefix."""
    if _HAS_NOTEBOOKUTILS:
        try:
            items = notebookutils.fs.ls(base_path)
            return [f.name for f in items if f.name.endswith('.csv')]
        except Exception as e:
            print(f"ERROR: Could not list {base_path}: {e}")
            return []
    else:
        try:
            return [f for f in os.listdir(base_path) if f.endswith('.csv')]
        except FileNotFoundError:
            print(f"ERROR: Path not found: {base_path}")
            return []

def discover_data_sources(base_path):
    """Scan CSV directory and classify files by naming convention."""
    dim_tables = []
    fact_tables_batch = []    # fact tables for Lakehouse/Warehouse
    fact_tables_event = []    # fact tables for Eventhouse (event-level)
    stream_tables = []        # stream tables for Eventhouse only

    files = _list_csv_files(base_path)
    if not files:
        print("Upload your CSV files to the Lakehouse Files area first.")
        return dim_tables, fact_tables_batch, fact_tables_event, stream_tables

    for f in sorted(files):
        table_name = f.replace('.csv', '')

        # ZT: Validate table name as safe identifier
        try:
            sanitize_identifier(table_name)
        except ValueError as e:
            print(f"  ⚠ ZT: Skipping file with unsafe name: {f!r} — {e}")
            log_audit_event("TABLE_REJECTED", table_name, str(e), "BLOCKED")
            continue

        if table_name.startswith('dim_'):
            dim_tables.append(table_name)
        elif table_name.startswith('stream_'):
            stream_tables.append(table_name)
        elif table_name.startswith('fact_'):
            # Heuristic: event-level facts contain time-series / clickstream data
            event_keywords = [
                '_events', '_interactions', '_clickstream', '_scans',
                '_alerts', '_location', '_handoff', '_administration',
                '_vital', '_assessments', '_escalation', '_transfers',
                '_integrity', '_inspections', '_rfi_', '_change_order',
                '_filing', '_contract', '_metadata', '_approval',
                '_delivery', '_regulatory', '_outage', '_rca',
                '_dispatch', '_nms_', '_dms_', '_mam_',
            ]
            if any(kw in table_name for kw in event_keywords):
                fact_tables_event.append(table_name)
            else:
                fact_tables_batch.append(table_name)
        else:
            # ZT: Unknown prefix — log and skip (least privilege)
            print(f"  ⚠ ZT: Skipping file with unrecognized prefix: {table_name}")
            log_audit_event("TABLE_REJECTED", table_name, "No recognized prefix (dim_/fact_/stream_)", "BLOCKED")
            continue

    return dim_tables, fact_tables_batch, fact_tables_event, stream_tables

# Run discovery — pass _FS_CSV_PATH (has file: prefix in Fabric, plain path locally)
DIM_TABLES, FACT_TABLES_BATCH, FACT_TABLES_EVENT, STREAM_TABLES = discover_data_sources(_FS_CSV_PATH)

# ZT: Validate all discovered table names
DIM_TABLES = validate_table_names(DIM_TABLES)
FACT_TABLES_BATCH = validate_table_names(FACT_TABLES_BATCH)
FACT_TABLES_EVENT = validate_table_names(FACT_TABLES_EVENT)
STREAM_TABLES = validate_table_names(STREAM_TABLES)

# Combined views for different targets
LAKEHOUSE_TABLES = DIM_TABLES + FACT_TABLES_BATCH   # All dims + batch facts → Lakehouse
WAREHOUSE_TABLES = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT  # All non-streaming → Warehouse
EVENTHOUSE_TABLES = FACT_TABLES_EVENT + STREAM_TABLES  # Events + streams → Eventhouse

log_audit_event("DATA_DISCOVERY", _RESOLVED_CSV_PATH,
    f"Found {len(DIM_TABLES)} dim, {len(FACT_TABLES_BATCH)} fact-batch, "
    f"{len(FACT_TABLES_EVENT)} fact-event, {len(STREAM_TABLES)} stream tables")

print(f"\n{'='*60}")
print(f"DATA SOURCE DISCOVERY — {INDUSTRY.upper()}")
print(f"{'='*60}")
print(f"Total CSV files:  {len(DIM_TABLES) + len(FACT_TABLES_BATCH) + len(FACT_TABLES_EVENT) + len(STREAM_TABLES)}")
print(f"Lakehouse target: {len(LAKEHOUSE_TABLES)} tables")
print(f"Warehouse target: {len(WAREHOUSE_TABLES)} tables")
print(f"Eventhouse target: {len(EVENTHOUSE_TABLES)} tables")

print(f"\nDimension tables ({len(DIM_TABLES)}):")
for t in DIM_TABLES: print(f"  • {t}")

print(f"\nFact tables — Batch/Lakehouse ({len(FACT_TABLES_BATCH)}):")
for t in FACT_TABLES_BATCH: print(f"  • {t}")

print(f"\nFact tables — Event/Eventhouse ({len(FACT_TABLES_EVENT)}):")
for t in FACT_TABLES_EVENT: print(f"  • {t}")

print(f"\nStreaming tables — Eventhouse ({len(STREAM_TABLES)}):")
for t in STREAM_TABLES: print(f"  • {t}")
print(f"\n{'─'*60}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 5: SCHEMA INSPECTION (preview any table)
# ════════════════════════════════════════════════════════════════════════
# Works in both Fabric runtime (spark) and local/VS Code (pandas fallback)

_USE_SPARK = False
try:
    spark  # noqa: F821 — defined in Fabric runtime
    _USE_SPARK = True
except NameError:
    import pandas as pd

def preview_table(table_name, base_path=None, rows=5):
    """Read a CSV and display schema + sample rows.
    Uses _FS_CSV_PATH (file:-prefixed in Fabric) for reliable path resolution."""
    if base_path is None:
        base_path = _FS_CSV_PATH
    path = f"{base_path}/{table_name}.csv"

    if _USE_SPARK:
        try:
            df = spark.read.option("header", True).option("inferSchema", True).csv(path)
        except Exception as e:
            print(f"\n⚠ Could not read: {path}")
            print(f"  Error: {e}")
            print("  → Verify the correct Lakehouse is attached as default to this notebook.")
            return None
        print(f"\n{'─'*60}")
        print(f"Table: {table_name}  |  {df.count()} rows  |  {len(df.columns)} columns")
        print(f"{'─'*60}")
        df.printSchema()
        df.show(rows, truncate=False)
        return df
    else:
        if not os.path.exists(path):
            print(f"\n⚠ File not found: {path}")
            return None
        df = pd.read_csv(path)
        print(f"\n{'─'*60}")
        print(f"Table: {table_name}  |  {len(df)} rows  |  {len(df.columns)} columns")
        print(f"{'─'*60}")
        print(f"\nSchema:")
        for col in df.columns:
            print(f"  |-- {col}: {df[col].dtype}")
        print(f"\nSample ({rows} rows):")
        print(df.head(rows).to_string(index=False))
        return df

# Preview the first dimension and first fact table
if DIM_TABLES:
    preview_table(DIM_TABLES[0])
if FACT_TABLES_BATCH:
    preview_table(FACT_TABLES_BATCH[0])

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# COMPLETE PIPELINE STEP
# ════════════════════════════════════════════════════════════════════════

try:
    log_artifact_created("00_Industry_Config", "Lakehouse", LAKEHOUSE_NAME)
    log_pipeline_event("CONFIG_COMPLETE", INDUSTRY, 
        f"{len(DIM_TABLES)} dim, {len(FACT_TABLES_BATCH)} fact-batch, {len(FACT_TABLES_EVENT)} fact-event, {len(STREAM_TABLES)} stream")
    pipeline_step_complete("00_Industry_Config", f"Discovered {len(DIM_TABLES) + len(FACT_TABLES_BATCH) + len(FACT_TABLES_EVENT) + len(STREAM_TABLES)} tables")
except NameError:
    pass

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CONFIG SUMMARY (exported to downstream notebooks via %run)
# ════════════════════════════════════════════════════════════════════════

CONFIG = {
    "industry":             INDUSTRY,
    "industry_label":       INDUSTRY_LABEL,
    "lakehouse_name":       LAKEHOUSE_NAME,
    "warehouse_name":       WAREHOUSE_NAME,
    "eventhouse_name":      EVENTHOUSE_NAME,
    "eventhouse_uri":       EVENTHOUSE_CLUSTER_URI,
    "eventhouse_db":        EVENTHOUSE_DATABASE,
    "ontology_name":        ONTOLOGY_NAME,
    "ontology_package":     ONTOLOGY_PACKAGE_PATH,
    "data_agent_name":      DATA_AGENT_NAME,
    "ops_agent_name":       OPS_AGENT_NAME,
    "semantic_model_name":  SEMANTIC_MODEL_NAME,
    "csv_base_path":        CSV_BASE_PATH,
    "lakehouse_schema":     LAKEHOUSE_SCHEMA,
    "warehouse_schema":     WAREHOUSE_SCHEMA,
    "dim_tables":           DIM_TABLES,
    "fact_tables_batch":    FACT_TABLES_BATCH,
    "fact_tables_event":    FACT_TABLES_EVENT,
    "stream_tables":        STREAM_TABLES,
    "lakehouse_tables":     LAKEHOUSE_TABLES,
    "warehouse_tables":     WAREHOUSE_TABLES,
    "eventhouse_tables":    EVENTHOUSE_TABLES,
}

print(json.dumps({k: v if not isinstance(v, list) else f"{len(v)} tables" for k, v in CONFIG.items()}, indent=2))

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# COMPLETE PIPELINE STEP
# ════════════════════════════════════════════════════════════════════════

try:
    log_artifact_created("00_Industry_Config", "Lakehouse", LAKEHOUSE_NAME)
    log_pipeline_event("CONFIG_COMPLETE", INDUSTRY, 
        f"{len(DIM_TABLES)} dim, {len(FACT_TABLES_BATCH)} fact-batch, {len(FACT_TABLES_EVENT)} fact-event, {len(STREAM_TABLES)} stream")
    pipeline_step_complete("00_Industry_Config", f"Discovered {len(DIM_TABLES) + len(FACT_TABLES_BATCH) + len(FACT_TABLES_EVENT) + len(STREAM_TABLES)} tables")
except NameError:
    pass

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PERSIST PIPELINE LOG TO LAKEHOUSE
# ════════════════════════════════════════════════════════════════════════

try:
    if FABRIC_RUNTIME and LAKEHOUSE_ABFS_BASE:
        persist_pipeline_log(spark, lakehouse_schema=LAKEHOUSE_SCHEMA, lakehouse_abfs_base=LAKEHOUSE_ABFS_BASE)
    elif FABRIC_RUNTIME:
        persist_pipeline_log(spark, lakehouse_schema=LAKEHOUSE_SCHEMA)
    else:
        print("⚠ Not in Fabric runtime — pipeline log not persisted")
except NameError:
    pass